In [5]:
import pandas as pd

# 1. Definimos la ruta local del archivo en el Lakehouse
# (Esta ruta interna es la que Fabric entiende por debajo)
ruta = "/lakehouse/default/Files/Lista-de-clientes-con-nombre-y-direccion.xlsx"

# 2. Leemos con Pandas (que es un teso para Excel)
# Nota: Si te pide openpyxl, ya lo instalamos en el paso anterior
df_pandas = pd.read_excel(ruta)

# 3. Lo convertimos a un DataFrame de Spark para poder usar la potencia de Fabric
df_final = spark.createDataFrame(df_pandas)

# 4. El momento de la gloria: mostrar los 1000 clientes
display(df_final)

StatementMeta(, eb3a1703-8e12-41f6-8b59-66f79b4d089e, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b3c22b32-0649-4b7f-97ae-f1c566d9511c)

In [7]:
# 1. Función rápida para limpiar los nombres de las columnas
# Esto cambia "Nombre completo" por "Nombre_completo" automáticamente
columnas_limpias = [c.replace(' ', '_').replace(';', '').replace('{', '').replace('}', '') for c in df_final.columns]

# 2. Le aplicamos los nuevos nombres al DataFrame
df_final = df_final.toDF(*columnas_limpias)

# 3. Ahora sí, intentamos guardar de nuevo
df_final.write.format("delta").mode("overwrite").saveAsTable("Clientes_Mokesoft_Limpio")

print("¡Ahora sí, Mauro! Tabla creada sin espacios prohibidos.")

StatementMeta(, eb3a1703-8e12-41f6-8b59-66f79b4d089e, 15, Finished, Available, Finished)

¡Ahora sí, Mauro! Tabla creada sin espacios prohibidos.


In [8]:
from pyspark.sql.functions import col, initcap, trim, lower

# 1. Quitamos duplicados (si hay dos filas exactamente iguales, dejamos solo una)
df_limpio = df_final.dropDuplicates()

# 2. Limpieza de texto:
# - trim: Quita espacios locos al principio o al final
# - initcap: Pone la primera letra en Mayúscula (Ej: "mauro" -> "Mauro")
# Vamos a aplicarlo a la columna 'Nombre_completo' (o como se llame en tu tabla)
# Ajusta el nombre de la columna según lo que viste arriba
df_limpio = df_limpio.withColumn("Nombre_completo", initcap(trim(col("Nombre_completo"))))

# 3. Guardamos esta versión como nuestra "Capa Plata" (Silver)
df_limpio.write.format("delta").mode("overwrite").saveAsTable("Clientes_Mokesoft_Silver")

print("¡Tarea cumplida, Mauro! Ya tienes la tabla limpia y profesional en la Capa Plata.")

StatementMeta(, eb3a1703-8e12-41f6-8b59-66f79b4d089e, 16, Finished, Available, Finished)

¡Tarea cumplida, Mauro! Ya tienes la tabla limpia y profesional en la Capa Plata.
